<a href="https://colab.research.google.com/github/LizethArista/Manejo_de_datos/blob/main/11_Aplicaciones_hash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicaciones de la tabla hash

Para el análisis de datos, es muy importante saber cómo extraer claves, valores y pares completos

- .keys(): Devuelve las claves (ej. nombres de estados)

- .values(): Devuelve los valores (ej. poblaciones)

- .items(): Devuelve pares (clave, valor), ideal para bucles for

Comprensión de diccionarios: Permite crear y filtrar diccionarios en una sola línea de código, similar a las listas de comprensión

In [1]:
# Diccionario de tasas de fecundidad por entidad
tasas_fecundidad = {
    "Chiapas": 2.8, "CDMX": 1.5, "Oaxaca": 2.6,
    "Nuevo León": 1.7, "Guerrero": 2.7, "Monterrey": 1.6 # Nota: Monterrey no es entidad, es error intencional
}

# Filtro solo las entidades con tasa > 2.0 usando comprensión de diccionarios
alta_fecundidad = {entidad: tasa for entidad, tasa in tasas_fecundidad.items() if tasa > 2.0}
print("Entidades con alta fecundidad:", alta_fecundidad)

# Aplica una transformación (ej. calcular crecimiento proyectado simple, es decir multiplicando por algun factor d crecimiento)

# Multiplicamos cada tasa por un factor de 1.05

proyeccion = {entidad: round(tasa * 1.05, 2) for entidad, tasa in tasas_fecundidad.items()}
print("Proyección:", proyeccion)

Entidades con alta fecundidad: {'Chiapas': 2.8, 'Oaxaca': 2.6, 'Guerrero': 2.7}
Proyección: {'Chiapas': 2.94, 'CDMX': 1.58, 'Oaxaca': 2.73, 'Nuevo León': 1.78, 'Guerrero': 2.84, 'Monterrey': 1.68}


# Agrupación y Agregación de Datos

En los Censos de Población, a menudo necesitamos calcular promedios o sumas condicionales (ej. ingreso promedio por nivel de escolaridad). Las tablas hash son el motor detrás de operaciones como groupby en Pandas

In [2]:
# AGRUPACIÓN PARA ANÁLISIS

# Datos simulados: Ingreso mensual (en miles de pesos) y nivel de escolaridad

datos_ingreso = [
    {"id": 1, "escolaridad": "Básica", "ingreso": 5.5},
    {"id": 2, "escolaridad": "Media Superior", "ingreso": 8.2},
    {"id": 3, "escolaridad": "Básica", "ingreso": 6.0},
    {"id": 4, "escolaridad": "Superior", "ingreso": 15.5},
    {"id": 5, "escolaridad": "Media Superior", "ingreso": 9.0},
    {"id": 6, "escolaridad": "Superior", "ingreso": 18.0}
]

# Calculamos el ingreso promedio por escolaridad usando tablas hash

# creo tablas hash para acumular suma y conteo
sumas_ingreso = {}
conteos_ingreso = {}

for registro in datos_ingreso:
    esc = registro["escolaridad"]
    ing = registro["ingreso"]

    # Acumulo la suma
    sumas_ingreso[esc] = sumas_ingreso.get(esc, 0.0) + ing
    # Acumulo el conteo
    conteos_ingreso[esc] = conteos_ingreso.get(esc, 0) + 1

# calculo los promedios en un nuevo diccionario
promedios_ingreso = {}
for esc in sumas_ingreso:
    promedios_ingreso[esc] = round(sumas_ingreso[esc] / conteos_ingreso[esc], 2)

print("Ingreso promedio (miles de pesos) por escolaridad:", promedios_ingreso)

Ingreso promedio (miles de pesos) por escolaridad: {'Básica': 5.75, 'Media Superior': 8.6, 'Superior': 16.75}


# Optimización de Búsquedas (Unión de Datos)

Cuando se trabaja con microdatos del INEGI, buscar si un ID existe en una lista usando if id in lista tiene un tiempo O(N)

Si usamos una tabla hash (o un set en Python, que es una tabla hash de solo claves), el tiempo es O(1). Esto es importante para identificar reunificación familiar

In [3]:
# OPTIMIZACIÓN DE BÚSQUEDAS (REUNIFICACIÓN)


# Lista de IDs de jefas de hogar migrantes ya establecidas
jefas_establecidas = ["MIG001", "MIG005", "MIG012", "MIG045", "MIG099"]

# Lista de IDs de familiares que acaban de llegar y solicitan reunificación
familiares_llegando = ["MIG005", "MIG002", "MIG099", "MIG100"]

# MALA PRÁCTICA (O(N) por búsqueda):
# for fam in familiares_llegando:
#     if fam in jefas_establecidas: ...

# BUENA PRÁCTICA: Convertir la lista a un Set (Tabla Hash de solo claves) O(1)
set_jefas = set(jefas_establecidas)

# Diccionario para guardar el resultado de la validación
validacion_reunificacion = {}

for fam in familiares_llegando:
    # La búsqueda en el set es instantánea O(1)
    if fam in set_jefas:
        validacion_reunificacion[fam] = "Aprobado: Vínculo existente"
    else:
        validacion_reunificacion[fam] = "Rechazado: Sin vínculo previo"

print("Resultados de validación:", validacion_reunificacion)

Resultados de validación: {'MIG005': 'Aprobado: Vínculo existente', 'MIG002': 'Rechazado: Sin vínculo previo', 'MIG099': 'Aprobado: Vínculo existente', 'MIG100': 'Rechazado: Sin vínculo previo'}


## Grupos quinquenales

Un grupo quinquenal es una categoría que agrupa a la población en intervalos de cinco años de edad (por ejemplo: 0 a 4 años, 5 a 9 años, 10 a 14 años)

Es el estándar principal utilizado por los institutos de estadística (como el INEGI en México) y organismos internacionales para analizar la estructura de una población

### Importancia en la demografía

• Construcción de pirámides poblacionales: Permite visualizar de forma clara y estandarizada la distribución por edad y sexo de un territorio, identificando si la población es mayoritariamente joven, adulta o envejecida

• Planificación de políticas públicas: Facilita el cálculo preciso de demandas futuras. Por ejemplo, el grupo de 5 a 9 años indica cuántas plazas escolares de primaria se necesitarán, mientras que el grupo de 60 a 64 años ayuda a proyectar el gasto en pensiones y salud

• Cálculo de tasas específicas: Permite medir fenómenos demográficos con mayor exactitud, como la fecundidad (analizando los grupos quinquenales de mujeres en edad fértil, de 15 a 49 años) o la mortalidad por etapas de la vida

• Comparabilidad internacional: Al ser un estándar global, permite comparar la evolución demográfica entre distintos países y épocas sin distorsiones

Las tablas hash son muy útil.es al momento de agrupar rgistros:




In [4]:

# GRUPOS QUINQUENALES DE EDAD
# Datos de entrada: edades de una muestra censal

edades_muestra = [2, 4, 5, 7, 8, 10, 12, 14, 15, 18, 2, 3, 6, 9, 11]

# Inicializo la tabla hash (diccionario) vacía
conteo_grupos = {}

# Itero sobre cada edad y clasificarla en su grupo quinquenal
for edad in edades_muestra:
    # Calcular el límite inferior del grupo quinquenal
    # Ejemplo: edad=7 → (7//5)*5 = 1*5 = 5 → grupo "5"
    grupo = (edad // 5) * 5

    # Acumulo en la tabla hash usando .get()
    # .get(grupo, 0) devuelve el valor actual o 0 si no existe
    conteo_grupos[grupo] = conteo_grupos.get(grupo, 0) + 1

# Ordeno el diccionario por clave (grupo de edad) para mejor visualización
conteo_ordenado = dict(sorted(conteo_grupos.items()))

print("Distribución por grupos quinquenales de edad:")
print("-" * 40)
for grupo, conteo in conteo_ordenado.items():
    print(f"Grupo {grupo:2d}-{grupo+4:2d} años: {conteo} individuos")

print("-" * 40)
print(f"Total de individuos procesados: {sum(conteo_grupos.values())}")

Distribución por grupos quinquenales de edad:
----------------------------------------
Grupo  0- 4 años: 4 individuos
Grupo  5- 9 años: 5 individuos
Grupo 10-14 años: 4 individuos
Grupo 15-19 años: 2 individuos
----------------------------------------
Total de individuos procesados: 15


Complejidad temporal: O(N), donde N es el número de edades. Cada inserción en la tabla hash toma O(1)

Complejidad espacial: O(K), donde K es el número de grupos únicos (en este caso, 4)

Si tuviéramos millones de registros censales, la búsqueda y actualización en el diccionario sería instantánea, mientras que en una lista tendríamos que recorrer todo el arreglo cada vez